## Notebook and df prep


In [1]:
#scikit-image is needed for RGB → CIELAB
%pip install scikit-image

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from matplotlib.colors import to_rgb
from skimage.color import rgb2lab
import requests
from io import BytesIO
from PIL import Image
from IPython.display import display
import time

In [3]:
# Read the raw csv-s into df
shades = pd.read_csv('../data/raw/allShades.csv')
monk = pd.read_csv('../data/raw/monk_scale.csv')

In [4]:
# Check if the dfs are successfully read using shape
print('shades:', shades.shape)
print('monk:', monk.shape)

shades: (6816, 13)
monk: (10, 2)


In [5]:


def get_image(url, tries=3):
    """Download an image; retry a few times; return None if it still fails."""
    if url.startswith('/'):
        url = 'https://www.sephora.com' + url
    for attempt in range(tries):
        try:
            response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
            return Image.open(BytesIO(response.content)).convert('RGB')
        except Exception:
            time.sleep(2)   # wait a moment, then try again
    return None

In [6]:
# set to True only when I want to download and look at images again
CHECK_IMAGES = False

In [7]:
# using the same setup as for the previous notebooks, just in case
pd.set_option('display.max_columns', None)

# show long text in full
pd.set_option('display.max_colwidth', None)

## Hex format check before conversion on shades

In [8]:
# True = correct hex format: '#' followed by exactly 6 hex characters (0-9, A-F)
shades['hex'].str.match(r'^#[0-9A-Fa-f]{6}$').value_counts(dropna=False)

hex
True    6816
Name: count, dtype: int64

## Conversion test on one hex code on shades

In [9]:
# one example, step by step
example = '#C99673'

rgb = to_rgb(example)
print('RGB (0-1):', rgb)

lab = rgb2lab([[rgb]])
print('CIELAB:', lab)

RGB (0-1): (0.788235294117647, 0.5882352941176471, 0.45098039215686275)
CIELAB: [[[66.11327084 14.69459236 25.91979377]]]


## Conversion from HEX to RGB to CIELAB on shades

In [10]:
# 1. hex -> RGB (0-1) for every shade: a list of 6,816 (r, g, b) tuples, made into a numpy array
rgb = np.array([to_rgb(h) for h in shades['hex']])
print('rgb:', rgb.shape)   # (6816, 3): one row per shade, 3 values each

# 2. rgb2lab expects an image (rows x columns x 3), so reshape to a 6816 x 1 "image"
rgb_image = rgb.reshape(-1, 1, 3)

# 3. convert all shades at once
lab = rgb2lab(rgb_image).reshape(-1, 3)
print('lab:', lab.shape)   # back to (6816, 3)

# 4. one column per value
shades['L_star'] = lab[:, 0]
shades['a_star'] = lab[:, 1]
shades['b_star'] = lab[:, 2]

rgb: (6816, 3)
lab: (6816, 3)


In [11]:
# three random shades with their new values
check = shades[['hex', 'L_star', 'a_star', 'b_star']].sample(3, random_state=42)
check

,hex,L_star,a_star,b_star
1886,#F4CDAA,84.894683,8.755810,22.567540
2943,#89583B,42.177062,16.993128,25.041605
838,#774D35,36.906700,14.910964,21.440008


In [12]:
# convert one of the sampled hex codes by itself, to compare with the table
h = check['hex'].iloc[0]
print(h, rgb2lab([[to_rgb(h)]]))

#F4CDAA [[[84.89468297  8.7558102  22.56753966]]]


## Conversion to LCh on shades

In [13]:
# chroma: distance from the gray center = how colorful (low = muted, high = saturated)
shades['chroma'] = np.sqrt(shades['a_star']**2 + shades['b_star']**2)

# hue: the angle of (a*, b*) in degrees = the direction (0° = red, 90° = yellow)
# % 360 turns any negative angle into its 0-360 equivalent
shades['hue'] = np.degrees(np.arctan2(shades['b_star'], shades['a_star'])) % 360

In [14]:
shades.loc[check.index, ['hex', 'L_star', 'a_star', 'b_star', 'chroma', 'hue']]

,hex,L_star,a_star,b_star,chroma,hue
1886,#F4CDAA,84.894683,8.755810,22.567540,24.206571,68.794619
2943,#89583B,42.177062,16.993128,25.041605,30.262986,55.839349
838,#774D35,36.906700,14.910964,21.440008,26.115336,55.182398


## Looking for potentially incorrect swatches on shades (i.e.: near white)

### Close to white shades, with chroma that is too low for skin color

In [15]:
# shades with a lightness close to white: likely background pixels, not foundation
near_white = shades[shades['L_star'] > 97]

near_white.shape

(43, 17)

In [16]:
# a look at them: which products, which hex, and the image to check
near_white[['brand', 'product', 'hex', 'L_star', 'imgSrc']].sort_values('L_star', ascending=False)

,brand,product,hex,L_star,imgSrc
432,NARS,Sheer Glow Foundation,#FEFEFE,99.654922,https://images.ulta.com/is/image/Ulta/2505579?$sm$
444,NARS,Sheer Glow Foundation,#FEFEFD,99.630069,https://images.ulta.com/is/image/Ulta/2538770?$sm$
446,NARS,Sheer Glow Foundation,#FEFEFD,99.630069,https://images.ulta.com/is/image/Ulta/2505594?$sm$
2774,Lancôme,Teint Idole Ultra Wear 24H Long Wear Foundation,#FEFEFD,99.630069,https://images.ulta.com/is/image/Ulta/2506588?$sm$
2143,L'Oréal,Infallible Pro-Matte Liquid Longwear Foundation,#FEFDFC,99.358363,https://images.ulta.com/is/image/Ulta/2547752?$sm$
2761,Lancôme,Teint Idole Ultra Wear 24H Long Wear Foundation,#FDFDFD,99.309587,https://images.ulta.com/is/image/Ulta/2538373?$sm$
6123,Erborian,BB Cream Tinted Moisturizer Broad Spectrum SPF20,#FDFDFD,99.309587,/productimages/sku/s2340156+sw-62.jpg
6122,Erborian,BB Cream Tinted Moisturizer Broad Spectrum SPF20,#FDFDFD,99.309587,/productimages/sku/s1936293+sw-62.jpg
6120,Erborian,BB Cream Tinted Moisturizer Broad Spectrum SPF20,#FDFDFD,99.309587,/productimages/sku/s1705839+sw-62.jpg
2770,Lancôme,Teint Idole Ultra Wear 24H Long Wear Foundation,#FDFDFD,99.309587,https://images.ulta.com/is/image/Ulta/2550814?$sm$


Near-white check: L* > 97  
- 43 shades with L* above 97, including the known #FDFCFB shades (Laura Mercier, Maybelline) and more from the same brands  
- the lowest one (L* 97.33, a Clinique product) is clearly a background error: white background, the product itself is much darker  
- a Maybelline porcelain shade (L* 97.6) is hard to judge by eye: very fair product on a white background  
- so 97 is too strict: faulty swatches may also sit below it

In [17]:
# the band just below the threshold: are these also background errors?
shades[(shades['L_star'] > 94) & (shades['L_star'] <= 97)][['brand', 'product', 'hex', 'L_star']].sort_values('L_star', ascending=False)

,brand,product,hex,L_star
2742,Lancôme,Teint Idole Ultra Wear 24H Long Wear Foundation,#FEF3ED,96.528204
2094,Maybelline,Fit Me Matte + Poreless Liquid Foundation,#FEF3EA,96.456731
2701,L'Oréal,Infallible Fresh Wear 24HR Foundation,#FDF2E2,95.946538
1258,Clinique,Almost Powder Makeup,#F4F3F2,95.890965
1259,Clinique,Almost Powder Makeup,#F4F3F2,95.890965
1255,Clinique,Almost Powder Makeup,#F4F3F1,95.866063
1256,Clinique,Almost Powder Makeup,#F4F3F1,95.866063
1257,Clinique,Almost Powder Makeup,#F4F3F1,95.866063
5732,Natasha Denona,Transformatte Pore Vanishing Matte Foundation,#FAF2E6,95.806553
1254,Clinique,Almost Powder Makeup,#F3F3F0,95.767002


Band L* 94–97  
- about 30 more shades  
- no clean gap: L* slides down slowly (only a small jump between 96.4 and 95.9)  
- so a cut-off on L* alone can't separate background errors from real, very fair foundations  

Plan: let the images decide, not a threshold  
1. candidate group: all shades with L* above 94  
2. re-measure each from its swatch image: the most common color that isn't the background  
3. compare old and new hex: clearly darker new color = old hex was the background, use the new one; about the same = real fair shade, keep the old one  
4. eye check on the ones that changed

Changed later: the chroma of the current hex decides which shades need fixing (see "Chroma decides" below).

In [18]:
candidates = shades[shades['L_star'] > 94]

candidates.shape

(81, 17)

### Test image: measuring the colors

In [19]:
# the first candidate's swatch image
if CHECK_IMAGES:

    url = candidates['imgSrc'].iloc[0]

    # Sephora links may start with '/', so they need the website in front
    if url.startswith('/'):
        url = 'https://www.sephora.com' + url
    print(url)

    # download the image; the User-Agent makes the request look like a normal browser
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)

    # open it as an image, in RGB
    img = Image.open(BytesIO(response.content)).convert('RGB')
    display(img)

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


this is a very good example, because it has a very little product color in the image

In [20]:
# shrink to 30 pixels wide (keeps the proportions), like The Pudding
if CHECK_IMAGES:

    small = img.resize((30, int(30 * img.height / img.width)))

    # reduce the whole image to its 12 most typical colors, less number of colors never picked up the "warm" shade that is the foundation
    quant = small.quantize(colors=12)

    # how many pixels have each of the 12 colors, most frequent first
    palette = quant.getpalette()
    for count, idx in sorted(quant.getcolors(), reverse=True):
        r, g, b = palette[idx * 3: idx * 3 + 3]
        print(count, f'#{r:02X}{g:02X}{b:02X}')

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


Measuring the colors (The Pudding's method, but with more colors)  
- 4 colors: white, black, gray (silver), near-white: no foundation color at all  
- 8 colors: #DCD5CD, slightly warm but almost gray: the foundation mixed with the silver around it  
- 12 colors: #E8DACB, the only warm (skin-like) color: closer, but still grayer than the real product, since the foundation covers only a small area  
- picked by hand (Digital Color Meter, sRGB): #F5D7B5 (an earlier pick with the wrong display setting gave #F1D5B6)  
- decision: use the hand-picked #F5D7B5 for this one; the automatic version would make it look more muted than it is

### Chroma decides

In [21]:
# near-white candidates, sorted by chroma: gray (background) first
candidates[['brand', 'product', 'hex', 'L_star', 'chroma']].sort_values('chroma')

,brand,product,hex,L_star,chroma
1651,Makeup Revolution,Conceal & Define Full Coverage Foundation,#F2F2F2,95.493559,0.005057
1876,Clinique,Superbalanced Makeup,#F9F9F9,97.925646,0.005167
1884,Clinique,Superbalanced Makeup,#F9F9F9,97.925646,0.005167
1878,Clinique,Superbalanced Makeup,#F9F9F9,97.925646,0.005167
1883,Clinique,Superbalanced Makeup,#F9F9F9,97.925646,0.005167
...,...,...,...,...,...
4651,HUDA BEAUTY,#FauxFilter Skin Finish Buildable Coverage Foundation Stick,#FCEFDD,95.013426,10.374435
6282,Bobbi Brown,Skin Long-Wear Fluid Powder Foundation SPF 20,#FEEBDC,94.165197,10.407346
2093,Maybelline,Fit Me Matte + Poreless Liquid Foundation,#FEEDDB,94.634279,11.157454
1267,Clinique,Beyond Perfecting Powder Foundation + Concealer,#FBEED9,94.597797,11.779798


In [22]:
# how many candidates per chroma value (rounded to whole numbers), from gray (0) upwards
candidates['chroma'].round().value_counts().sort_index()

chroma
0.0     20
1.0     27
2.0      2
4.0      1
5.0      2
6.0      2
7.0     10
8.0      5
9.0      3
10.0     6
11.0     1
12.0     2
Name: count, dtype: int64

Which candidates are wrong: chroma decides  
- no clean gap in L*, but a clear one in chroma: 49 shades at 0–2 (gray = background), none at 3, 32 shades at 4–12 (warm = real fair shades)  
- rule: chroma < 3 = needs fixing (needs_fix, 49 shades); the 32 others are kept as they are

In [23]:
# the candidates that need fixing: gray hex (chroma < 3) = background, not foundation
needs_fix = candidates[candidates['chroma'] < 3]

needs_fix.shape   # should be (49, 17)

(49, 17)

### Edge checks

In [24]:
# edge check: the shades closest to the gap (rounded chroma 2 and 4)
edges = candidates[candidates['chroma'].round().isin([2, 4])]

edges[['brand', 'product', 'hex', 'L_star', 'chroma', 'imgSrc']]

,brand,product,hex,L_star,chroma,imgSrc
1254,Clinique,Almost Powder Makeup,#F3F3F0,95.767002,1.540568,https://images.ulta.com/is/image/Ulta/2160689?$tn$
2741,Lancôme,Teint Idole Ultra Wear 24H Long Wear Foundation,#F6EFE8,94.802987,4.370050,https://images.ulta.com/is/image/Ulta/2550815sw?$50px$
6546,Josie Maran,Argan Daily Defense Lotion SPF 30,#FCF9F6,98.073875,1.852918,/productimages/sku/s2403319+sw.jpg


In [25]:
if CHECK_IMAGES:

    for _, row in edges.iterrows():
        print(row['brand'], '|', row['product'], '|', row['hex'], '| chroma', round(row['chroma'], 1))

        img = get_image(row['imgSrc'])   # retries, returns None if the download fails
        if img is None:
            print('   image could not be downloaded')
        else:
            display(img)                                       # the swatch image
        display(Image.new('RGB', (100, 100), row['hex']))     # its current hex as a color patch

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


In [26]:
# what did the server actually send back for the third edge shade?
if CHECK_IMAGES:

    row = edges.iloc[2]
    url = row['imgSrc']
    if url.startswith('/'):
        url = 'https://www.sephora.com' + url

    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
    print(row['brand'], '|', url)
    print('status:', response.status_code)
    print('type:', response.headers.get('Content-Type'))

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


Edge checks (closest to the gap)  
- Clinique, chroma 1.5: the hex is the white sponge, not the foundation → needs fixing ✔  
- Lancôme, chroma 4.3: the foundation itself, very pale → keep ✔  
- Josie Maran, chroma ~1.8: download refused by Sephora (403); opened with a VPN: the image is a flat, near-white color chip  
- so the line at chroma 3 holds

Sephora near-whites (5 of the 49)  
- Sephora's swatch images vary: some are flat color chips, some are small product photos  
- Josie Maran: a flat, near-white chip → no background that could "win"; the hex is exactly the color Sephora showed → kept as it is  
- Erborian BB Cream (4 shades): small product photos with a white background, a very light product and a silver cap; gray hex (chroma ≈ 0) → likely background errors → fix  
- Python downloads are refused (403); fix by hand in the browser (VPN on, Digital Color Meter, sRGB), unless the download works with the VPN on

### Dark shades, with chroma that is too low for skin color

In [27]:
# dark and gray: possible black-background errors (the mirror image of the near-whites)
shades[(shades['L_star'] < 25) & (shades['chroma'] < 3)][['brand', 'product', 'hex', 'L_star', 'chroma', 'imgSrc']]

,brand,product,hex,L_star,chroma,imgSrc


it seems that I do not have incorrect hex codes in the darker colors, but just to be on the safe side I will run another check

In [28]:
# the 10 darkest shades overall, with their chroma
shades.sort_values('L_star')[['brand', 'product', 'hex', 'L_star', 'chroma']].head(10)

,brand,product,hex,L_star,chroma
4298,beautyblender,Bounce™ Liquid Whip Long Wear Foundation,#3A2115,15.704126,16.821039
3225,Milani,Screen Queen Foundation,#2F2828,16.869733,3.447958
3068,Juvia's Place,I Am Magic Foundation,#382725,17.578300,8.852318
3224,Milani,Screen Queen Foundation,#392928,18.404871,8.103221
49,Anastasia Beverly Hills,Luminous Foundation,#3E2822,18.616082,12.424966
3066,Urban Decay Cosmetics,Stay Naked Weightless Liquid Foundation,#392B27,18.988977,7.663015
2868,Morphe,Fluidity Full-Coverage Foundation,#3E2927,19.041843,10.846301
4002,Dose Of Colors,Meet Your Hue Foundation,#3D2A24,19.121762,10.806831
3700,ZOEVA,Authentik Skin Natural Luminous Foundation,#452722,19.358245,16.344487
48,Anastasia Beverly Hills,Luminous Foundation,#422922,19.473716,14.215305


In [29]:
# among the darker shades (L* under 40), the 10 least colorful
shades[shades['L_star'] < 40].sort_values('chroma')[['brand', 'product', 'hex', 'L_star', 'chroma']].head(10)

,brand,product,hex,L_star,chroma
3225,Milani,Screen Queen Foundation,#2F2828,16.869733,3.447958
4589,FENTY BEAUTY by Rihanna,Pro Filt'r Hydrating Longwear Foundation,#635550,37.351792,6.976852
3066,Urban Decay Cosmetics,Stay Naked Weightless Liquid Foundation,#392B27,18.988977,7.663015
2869,Morphe,Fluidity Full-Coverage Foundation,#3B2D28,19.882661,7.919055
3224,Milani,Screen Queen Foundation,#392928,18.404871,8.103221
3067,Urban Decay Cosmetics,Stay Naked Weightless Liquid Foundation,#463631,24.201119,8.563806
3068,Juvia's Place,I Am Magic Foundation,#382725,17.578300,8.852318
6010,TOM FORD,Traceless Foundation Stick,#5A4642,31.687405,9.690018
4102,FENTY BEAUTY by Rihanna,Pro Filt'r Soft Matte Longwear Foundation,#66524C,36.708770,9.862630
2809,L.A. Girl,Pro.Matte Foundation,#534139,29.197701,10.132150


In [30]:
# I want to double check the Milani product as that has the lowest chroma and there is a bigger jump until the next product
if CHECK_IMAGES:

    row = shades.loc[3225]
    print(row['brand'], '|', row['product'], '|', row['hex'], '|', row['imgSrc'])
    display(get_image(row['imgSrc']))
    display(Image.new('RGB', (60, 60), row['hex']))

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


Dark-gray check (black-background errors)  
- L* < 25 and chroma < 3: empty  
- double check: the 10 darkest shades (L* 15.7–19.5, chroma 3.4–16.8) and the 10 least colorful dark shades (L* < 40: chroma 3.44–10.1)  
- lowest: Milani Screen Queen Foundation, #2F2828, chroma 3.45, with a jump to the next (6.97, Fenty Beauty); eye check: a real, muted deep brown foundation → kept  
- so no black-background errors; both ends of the scale are checked

## Fixing the swatches

### Automatic re-measuring:

In [31]:
if CHECK_IMAGES:

    def measure_swatch(url, n_colors=12, min_chroma=3):
        """Download a swatch image and return its most frequent colorful (non-gray) color."""
        if url.startswith('/'):
            url = 'https://www.sephora.com' + url
        try:
            response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
            img = Image.open(BytesIO(response.content)).convert('RGB')
        except Exception:
            return None, None, 'download failed'

        # shrink and reduce to n_colors, then count pixels per color
        small = img.resize((30, max(1, int(30 * img.height / img.width))))
        quant = small.quantize(colors=n_colors)
        palette = quant.getpalette()
        total = small.width * small.height

        # go through the colors from most to least frequent; keep the first colorful one
        for count, idx in sorted(quant.getcolors(), reverse=True):
            r, g, b = palette[idx * 3: idx * 3 + 3]
            lab = rgb2lab([[[r / 255, g / 255, b / 255]]])[0][0]
            chroma = np.sqrt(lab[1]**2 + lab[2]**2)
            if chroma >= min_chroma:
                return f'#{r:02X}{g:02X}{b:02X}', count / total, 'ok'

        return None, None, 'no colorful color'


    # run it on every shade that needs fixing
    results = []
    for _, row in needs_fix.iterrows():
        new_hex, share, status = measure_swatch(row['imgSrc'])
        results.append({'imgSrc': row['imgSrc'], 'brand': row['brand'], 'product': row['product'],
                        'old_hex': row['hex'], 'new_hex': new_hex, 'share': share, 'status': status})

    remeasured = pd.DataFrame(results)

    # flag for a manual check: download failed, no colorful color, or the colorful one covers under 5% of the image
    remeasured['manual_check'] = (remeasured['status'] != 'ok') | (remeasured['share'] < 0.05)

    display(remeasured['manual_check'].value_counts())

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


In [32]:
if CHECK_IMAGES:

    remeasured[remeasured['manual_check']].to_csv('../data/manual/remeasured_manual_check.csv', index=False)

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')    

Image downloads skipped (CHECK_IMAGES = False)


once again exported the results into CSV so I could add the colors manually where needed

### Eye check of the automatic results:

In [33]:
# eye checking the results
# for each automatic shade, in this order:
# Number than can be resused for collecting the problematic shades
# A text line: brand | product | new hex
# The swatch/product image
# A small square in the new color
if CHECK_IMAGES:

    auto_rows = remeasured[~remeasured['manual_check']].reset_index(drop=True)

    for i, row in auto_rows.iterrows():
        print(i, '|', row['brand'], '|', row['product'], '|', row['new_hex'])

        img = get_image(row['imgSrc'])   # retries, returns None if the download fails
        if img is None:
            print('   image could not be downloaded')
        else:
            display(img)
        display(Image.new('RGB', (60, 60), row['new_hex']))

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


In [34]:
if CHECK_IMAGES:

    display((~remeasured['manual_check']).sum())

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


In [35]:
# the numbers of the automatic results that look wrong (from the printed lines)
if CHECK_IMAGES:

    bad = [0, 4, 6, 7, 8, 9, 13, 14, 15, 16, 17, 19, 27 ]   # replace with my numbers

    # export them with all the columns, ready for hand-picking
    to_fix = auto_rows.loc[bad, ['imgSrc', 'brand', 'product', 'old_hex', 'new_hex']].copy()
    to_fix['final_hex'] = ''
    to_fix['method'] = ''
    to_fix.to_csv('../data/manual/auto_rejected.csv', index=False)

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


### Re-check of the kept candidates:

In [36]:
# double check: Maybelline, Dream Radiant Liquid Foundation, #FEF7EB: still above L* 97 after the final conversion;
# chroma 6.6 kept it, but it looks too light -> led to re-checking all kept candidates

if CHECK_IMAGES:

    row = shades.loc[3257]
    print(row['brand'], '|', row['product'], '|', row['hex'], '|', row['imgSrc'])
    display(get_image(row['imgSrc']))
    display(Image.new('RGB', (60, 60), row['hex']))

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


In [37]:
if CHECK_IMAGES:

    kept = candidates[candidates['chroma'] >= 3].reset_index()

    for i, row in kept.iterrows():
        print(i, '|', row['brand'], '|', row['product'], '|', row['hex'])

        img = get_image(row['imgSrc'])   # retries, returns None if the download fails
        if img is None:
            print('   image could not be downloaded')
        else:
            display(img)
        display(Image.new('RGB', (60, 60), row['hex']))

        time.sleep(0.5)   # a short pause, so the server isn't flooded

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


In [38]:
# the kept candidates from Sephora (image links starting with '/'), with full links to open in the browser
if CHECK_IMAGES:

    sephora_kept = kept[kept['imgSrc'].str.startswith('/')].copy()
    sephora_kept['full_url'] = 'https://www.sephora.com' + sephora_kept['imgSrc']

    pd.set_option('display.max_colwidth', None)
    display(sephora_kept[['brand', 'product', 'hex', 'full_url']])

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


In [39]:
# numbers of the kept shades whose patch looks wrong (from the printed lines)
if CHECK_IMAGES:

    bad_kept = [0, 4, 5, 6, 7, 8, 12, 13, 14, 16, 17, 19, 21, 22, 23, 24,25, 26, 27, 28, 29, 30, 31  ]  

    kept_to_fix = kept.loc[bad_kept, ['imgSrc', 'brand', 'product', 'hex']].rename(columns={'hex': 'old_hex'})
    kept_to_fix['final_hex'] = ''
    kept_to_fix['method'] = ''
    kept_to_fix.to_csv('../data/manual/kept_rechecked.csv', index=False)

else:
    print('Image downloads skipped (CHECK_IMAGES = False)')

Image downloads skipped (CHECK_IMAGES = False)


In [40]:
# shade details for the rows without a usable photo, to search for other photos
not_available = pd.read_csv('../data/manual/kept_rechecked_filled_in.csv')
not_available = not_available[not_available['method'] == 'prod_photo_not_available']

shades.loc[shades['imgSrc'].isin(not_available['imgSrc']), ['brand', 'product', 'name', 'specific', 'description', 'hex']]

,brand,product,name,specific,description,hex
2430,It Cosmetics,Bye Bye Lines Sheer Liquid Foundation,Fair,NaN,Fair (porcelain) OUT OF STOCK,#FAEDE0
2811,Morphe,Fluidity Full-Coverage Foundation,Neutral,F1.20,F1.20 Neutral (fairest w/ slightly yellow undertones),#FAECDB
3962,Dose Of Colors,Meet Your Hue Foundation,Fair,101,101 Fair (very fair w/ neutral undertones),#FCECE3


### Special cases

Revolution Conceal & Define Full Coverage Foundation: frosted bottle with a large label; little of the product visible  
- frosted glass makes the color look lighter and grayer than the product itself  
- picked by hand where the product is visible; a shade that can't be read stays dropped

In [41]:
shades.loc[shades['brand'] == 'Josie Maran', ['product', 'name', 'specific', 'description', 'hex']]

,product,name,specific,description,hex
6138,Vibrancy Argan Oil Foundation Fluid,Dynamic,RG5,Dynamic RG5 for fair skintones with neutral undertones,#E7CAC0
6139,Vibrancy Argan Oil Foundation Fluid,Supernatural,RG10,Supernatural RG10 for fair-light skintones with neutral undertones,#E4C4B7
6140,Vibrancy Argan Oil Foundation Fluid,Mystical,RG30,Mystical RG30 for light-medium skintones with neutral undertones,#D5B19F
6141,Vibrancy Argan Oil Foundation Fluid,Juicy,RG50,Juicy RG50 for medium skintones with neutral/neutral red undertones - Selected,#CBAE9F
6142,Vibrancy Argan Oil Foundation Fluid,Reverent,RG75,Reverent RG75 for tan-rich skintones with neutral undertones,#A97F6B
6143,Vibrancy Argan Oil Foundation Fluid,Magical,RG80,Magical RG80 for rich skintones with neutral/neutral red undertones,#9E715F
6144,Vibrancy Argan Oil Foundation Fluid,Unstoppable,RG85,Unstoppable RG85 for deep skintones with neutral/neutral red undertones,#8F6459
6484,Argan Matchmaker Serum Foundation,Fair Light,NaN,Fair/ Light for the fairest porcelain skin with pink undertones to light skin with pink undertones - Selected,#E9D5C6
6485,Argan Matchmaker Serum Foundation,Medium Dark,NaN,Medium/ Dark for medium tan skin all the way to the deepest skin tones,#CF9D7E
6546,Argan Daily Defense Lotion SPF 30,Clean at Sephora,NaN,- Selected,#FCF9F6


after discovering that Argan Daily Defense Lotion SPF 30 Josie Maran is a sunscreen (one row is the tinted shade, the other is not) I realized I need to make sure at this shade that I only have products in my analysis that have a shade range, untinted and tinted sunscreen does not make sense for my analysis

## Applying the corrections

In [42]:
# part 2: my manual work, always read fresh from the three review files
manual_fixes = pd.concat([
    pd.read_csv('../data/manual/remeasured_manual_check_filled_in.csv')[['imgSrc', 'old_hex', 'final_hex', 'method']],
    pd.read_csv('../data/manual/auto_rejected_filled_in.csv')[['imgSrc', 'old_hex', 'final_hex', 'method']],
    pd.read_csv('../data/manual/kept_rechecked_filled_in.csv')[['imgSrc', 'old_hex', 'final_hex', 'method']],
], ignore_index=True)

# part 1: the automatic results: rebuilt from the downloads, or taken from the saved corrections
if CHECK_IMAGES:
    auto_fixes = auto_rows.drop(index=bad)[['imgSrc', 'old_hex', 'new_hex']].rename(columns={'new_hex': 'final_hex'})
    auto_fixes['method'] = 'auto'
else:
    saved = pd.read_csv('../data/manual/hex_corrections.csv')
    auto_fixes = saved[saved['method'] == 'auto']

# together: one corrections table, saved
corrections = pd.concat([auto_fixes, manual_fixes], ignore_index=True)
corrections = corrections.drop_duplicates(subset='imgSrc', keep='last')   # a shade corrected twice: the manual fix (last) wins
corrections.to_csv('../data/manual/hex_corrections.csv', index=False)

corrections['method'].value_counts(dropna=False)

method
hand                        52
auto                        16
prod_photo_not_available     3
excluded_mixer               1
dropped                      1
Name: count, dtype: int64

In [43]:
# shades without a usable color (dropped) or not a wearable shade (excluded_mixer) are removed
removed = corrections.loc[corrections['method'].isin(['dropped', 'excluded_mixer']), 'imgSrc']
shades = shades[~shades['imgSrc'].isin(removed)].copy()

# hex_final: the corrected hex where one exists, otherwise the original hex
fix_map = dict(zip(corrections['imgSrc'], corrections['final_hex']))
shades['hex_final'] = shades['imgSrc'].map(fix_map).fillna(shades['hex'])

print('shades left:', shades.shape[0])
print('corrected:', (shades['hex_final'] != shades['hex']).sum())

shades left: 6814
corrected: 66


### Scope rule

In [44]:
# number of shades per product, and the products below the minimum
shades_per_product = shades.groupby(['brand', 'product']).size()
shades_per_product[shades_per_product < 5].sort_values()

brand                      product                                                                 
Dr. Jart+                  Black Label Detox BB Beauty Balm                                            1
Urban Decay Cosmetics      Naked Skin Ultra Definition Pressed Finishing Powder                        1
Lancôme                    Bienfait Teinté Beauty Balm Sunscreen Broad Spectrum SPF 30                 1
bareMinerals               bareSkin® Pure Brightening Serum Foundation Broad Spectrum SPF 20           1
Black Up                   Full Coverage Cream Foundation                                              1
Clinique                   Age Defense BB Cream Broad Spectrum SPF 30                                  1
Benefit Cosmetics          Hello Happy Soft Blur Foundation                                            1
Marc Jacobs Beauty         Shameless Youthful-Look 24H Foundation SPF 25                               1
Antonym                    Certified Organic Baked Foundatio

In [45]:
# how many products have 1, 2, 3, … shades
shades_per_product.value_counts().sort_index().head(5)

1     9
2     7
3    10
4     8
5    12
Name: count, dtype: int64

Scope rule: minimum shade range  
- products with fewer than 3 shades are excluded: 9 products with 1 shade, 7 with 2 → 16 products, 23 shades (of ~6,800)  
- reason: 1–2 shades = universal or sheer products (BB balms, tinted sunscreens) or minis, not a shade range  
- products with 3–5 shades stay: small ranges are part of what the project analyzes  
- product type (e.g. SPF vs BB) is a separate question, for the product-type rule in dbt

In [46]:
# remove products with fewer than 3 shades (not a shade range)
shades_per_product = shades.groupby(['brand', 'product'])['imgSrc'].transform('count')
shades = shades[shades_per_product >= 3].copy()

shades.shape

(6791, 18)

### Final conversion

In [47]:
# final conversion: from the corrected colors (hex_final)
rgb = np.array([to_rgb(h) for h in shades['hex_final']])
lab = rgb2lab(rgb.reshape(-1, 1, 3)).reshape(-1, 3)

shades['L_star'] = lab[:, 0]
shades['a_star'] = lab[:, 1]
shades['b_star'] = lab[:, 2]

shades['chroma'] = np.sqrt(shades['a_star']**2 + shades['b_star']**2)
shades['hue'] = np.degrees(np.arctan2(shades['b_star'], shades['a_star'])) % 360

# check again
(shades['L_star'] > 97).sum()

np.int64(0)

In [48]:
shades.loc[shades['L_star'] > 97, ['brand', 'product', 'hex', 'hex_final', 'L_star', 'chroma']]

,brand,product,hex,hex_final,L_star,chroma


## Monk tones

In [49]:
# CIELAB for the 10 Monk tones, with the same tool as the shades
monk_rgb = np.array([to_rgb(h) for h in monk['hex']])
monk_lab = rgb2lab(monk_rgb.reshape(-1, 1, 3)).reshape(-1, 3)
monk['L_star'] = monk_lab[:, 0]

monk[['monk_tone', 'hex', 'L_star']]

,monk_tone,hex,L_star
0,1,#f6ede4,94.211092
1,2,#f3e7db,92.274769
2,3,#f7ead0,93.091096
3,4,#eadaba,87.573317
4,5,#d7bd96,77.902219
5,6,#a07e56,55.142877
6,7,#825c43,42.470001
7,8,#604134,30.678265
8,9,#3a312a,21.069480
9,10,#292420,14.610244


In [50]:
# distance in lightness from every shade to every Monk tone: a table of 6,800 x 10 differences
distances = np.abs(shades['L_star'].values[:, None] - monk['L_star'].values[None, :])

# for each shade, the Monk tone with the smallest difference
shades['monk_tone'] = monk['monk_tone'].values[distances.argmin(axis=1)]

shades['monk_tone'].value_counts().sort_index()

monk_tone
1       30
2      171
3       30
4      807
5     3124
6     1585
7      685
8      292
9       64
10       3
Name: count, dtype: int64

we can already see some interesting trends showing after the conversion of the Monk scale, however a few important things to note:
- the Monk tones aren’t evenly spaced in L* -> if the gap between two tones is wider, more shades naturally fall into them  
- the ends collect everything beyond them  
- it’s the January 2021 market, as shown in online images  

## Exporting the conversions

In [51]:
# shades with all original columns plus hex_final, L*, a*, b*, chroma, hue and monk_tone
shades.to_csv('../data/processed/shades_color.csv', index=False)

# the Monk scale with its L* values, for dbt later (as a seed)
monk.to_csv('../data/processed/monk_scale_lab.csv', index=False)